[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/your-repository-path/your-notebook.ipynb)

In [ ]:
!git clone https://github.com/protosome/convergent_overlaps_aa_change.git

In [ ]:
%cd /content/convergent_overlaps_aa_change/s4pred

In [ ]:
!wget http://bioinfadmin.cs.ucl.ac.uk/downloads/s4pred/weights.tar.gz
!tar -xvzf weights.tar.gz

In [ ]:
%cd /content/convergent_overlaps_aa_change

In [ ]:
pip install biopython

In [ ]:

import torch
import tensorflow as tf
import numpy as np
import pandas as pd
import json
from Bio import pairwise2
from Bio.Seq import Seq
from Bio.Seq import CodonTable
import itertools as it
import pandas as pd
import math
import random
import pickle
from protsub_matrix import prot_sub_matrix, calculate_protsub_similarity
from blosum62_matrix import blosum62_matrix, calculate_blosum62_similarity
#sys.modules['__main__'].__dict__['TransformerModel'] = TransformerModel
from running_s4pred import predict_secondary_structure # Thisloads the s4pred function to run as a subprocess, outputting only the structure prediction sequence
from running_s4pred_batch import predict_secondary_structure_batch # Thisloads the s4pred function to run as a subprocess, outputting only the structure prediction sequence
from transformer_encoder_model import TransformerModel, SinusoidalPositionalEncoding, TransformerBlock

In [ ]:

# Identify available GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)
torch.cuda.get_device_name(0)

# Tokenization and text vectorization
max_length = 315  # max len of the overlap
vocab_size = 27

# Define a function to load the tokenizer
def load_tokenizer(filename):
    with open(filename, 'rb') as file:
        tokenizer = pickle.load(file)
    return tokenizer

concat_tokenizer = load_tokenizer('concat_tokenizer.pkl')
overlap_tokenizer = load_tokenizer('overlap_tokenizer.pkl')

# Function to tokenize sentences using TensorFlow's tokenizer
def tokenize(sentences):
    tokenizer = tf.keras.preprocessing.text.Tokenizer(num_words=vocab_size, filters='')
    tokenizer.fit_on_texts(sentences)  # Fit the tokenizer on the combined texts
    return tokenizer

# concat_tokenizer = tokenize(concat_test_model)
# overlap_tokenizer = tokenize(overlap_test_model)

# Function to vectorize sentences using the fitted tokenizer
def vectorize(tokenizer, sentences):
    seqs = tokenizer.texts_to_sequences(sentences)  # Convert texts to sequences of integers
    return tf.keras.preprocessing.sequence.pad_sequences(seqs, maxlen=max_length, padding='post')  # Pad sequences


In [ ]:

####################################################################################
### Function to predict the overlapping dna sequence for two aa's
####################################################################################

def predict_overlapping_sequence(trained_model, aa_sequences, concat_tokenizer, overlap_tokenizer):
    
    def translate_to_dna_with_all_options(aa_sequence: str) -> list:
    
        # Codons and their frequencies for each amino acid based on the E. coli table
        back_translation_code_with_all_options = {
            'A': [('GCG', 0.27), ('GCT', 0.26), ('GCC', 0.26), ('GCA', 0.21)],
            'C': [('TGC', 0.53), ('TGT', 0.47)],
            'D': [('GAT', 0.63), ('GAC', 0.37)],
            'E': [('GAA', 0.68), ('GAG', 0.32)],
            'F': [('TTT', 0.58), ('TTC', 0.42)],
            'G': [('GGC', 0.35), ('GGT', 0.32), ('GGG', 0.25), ('GGA', 0.08)],
            'H': [('CAT', 0.56), ('CAC', 0.44)],
            'I': [('ATT', 0.48), ('ATC', 0.39), ('ATA', 0.14)],
            'K': [('AAA', 0.74), ('AAG', 0.26)],
            'L': [('CTG', 0.43), ('CTT', 0.13), ('CTC', 0.13), ('TTA', 0.14), ('CTA', 0.07), ('TTG', 0.13)],
            'M': [('ATG', 1.00)],
            'N': [('AAC', 0.60), ('AAT', 0.40)],
            'P': [('CCG', 0.52), ('CCA', 0.19), ('CCT', 0.16), ('CCC', 0.13)],
            'Q': [('CAG', 0.66), ('CAA', 0.34)],
            'R': [('CGT', 0.36), ('CGC', 0.36), ('CGG', 0.11), ('AGA', 0.08), ('AGG', 0.05), ('CGA', 0.04)],
            'S': [('AGC', 0.24), ('TCC', 0.24), ('TCT', 0.17), ('TCG', 0.15), ('TCA', 0.14), ('AGT', 0.15)],
            'T': [('ACC', 0.36), ('ACA', 0.28), ('ACG', 0.25), ('ACT', 0.11)],
            'V': [('GTG', 0.46), ('GTT', 0.28), ('GTC', 0.15), ('GTA', 0.11)],
            'W': [('TGG', 1.00)],
            'Y': [('TAT', 0.59), ('TAC', 0.41)],
            '*': [('TAA', 0.61), ('TGA', 0.30), ('TAG', 0.09)]
        }
    
        def choose_codon_based_on_frequency(codons):

            # Extract codon names and their frequencies
            codon_names = [codon for codon, _ in codons]
            codon_freqs = [freq for _, freq in codons]
            
            # Normalize the frequencies to ensure they sum up to 1
            total_frequency = sum(codon_freqs)
            normalized_freqs = [freq/total_frequency for freq in codon_freqs]
            
            # Randomly select a codon based on the frequency distribution
            chosen_codon = np.random.choice(codon_names, p = normalized_freqs)
            
            return [chosen_codon]
    
        # For each amino acid in the sequence, choose a codon based on its frequency
        list_of_list_of_codons = [choose_codon_based_on_frequency(back_translation_code_with_all_options[aa]) for aa in aa_sequence]
        
        # Combine the codons to form the nucleotide sequence
        list_of_combinations = [''.join(combination) for combination in it.product(*list_of_list_of_codons)]
        
        return list_of_combinations

    def translate(model, concat_tokenizer, overlap_tokenizer, text, device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')):
        # Set the model to evaluation mode
        model.train()

        #model.eval()
    
        vectorized_input = vectorize(concat_tokenizer, [text])
        vectorized_input = torch.tensor(vectorized_input, dtype=torch.long).to(device)
    
        # Perform inference
        with torch.no_grad():
            output = model(vectorized_input)
    
        # Get the predicted classes
        _, predicted_classes = torch.max(output, dim=-1)
        predicted_classes = predicted_classes.cpu().numpy()
    
        # Function to reverse the predicted output back into text
        def decode_predictions(predicted_classes, tokenizer):
            # Get the index to word mapping from the tokenizer
            index_word = tokenizer.index_word
        
            # Convert indices to words
            predicted_text = ' '.join([index_word.get(index, '') for index in predicted_classes[0]])
        
            return predicted_text
    
        # Decode the predicted classes into text
        predicted_text = decode_predictions(predicted_classes, overlap_tokenizer)
    
        return predicted_text


    def convert_chars_for_translated_overlap(string):
        string = string.replace("no overlap", "#", 1)  # Replace first occurrence of "no overlap" with "#"
        string = string.replace("overlap", "#", 1)  # Replace first occurrence of "overlap" with "#"
        string = string.replace("no", "#", 1)  # Replace first occurrence of "no" with "#"
        if string == "#":
            return string
        new_string = ""
        for char in string:
            if char.lower() == "b":
                new_string += "A"
            elif char.lower() == "j":
                new_string += "G"
            elif char.lower() == "o":
                new_string += "T"
            elif char.lower() == "u":
                new_string += "C"
            else:
                new_string += char
        return new_string.replace(" ", "")

    # Running the above sequence through the model, using the tokenizer run on the data from the original model training process
    translated_overlap = translate(trained_model, concat_tokenizer, overlap_tokenizer, aa_sequences)
    
    overlap_output_from_model = convert_chars_for_translated_overlap(translated_overlap)
    
    # Generate the rc to include in the next small df
    reverse_complement_overlap = Seq(overlap_output_from_model)
    reverse_complement_overlap = str(reverse_complement_overlap.reverse_complement())
    
    # Create a DataFrame with two rows: original and reversed sequences
    # Note, the rc sequence actually comes first here, followed by the model output overlap sequence
    output_forward_reverse = pd.DataFrame({"overlap_sequence": [reverse_complement_overlap, overlap_output_from_model]})
    
    # Splitting and collapsing the sequences
    sequence_list = aa_sequences.split()
    collapsed_sequence = ''.join(sequence_list)
    
    # Splitting the sequence at the asterisk and keeping the asterisk
    split_sequences_with_asterisk = [seq + '*' for seq in collapsed_sequence.split('*') if seq]
    
    # Creating a dataframe from the sequences
    df_sequences = pd.DataFrame(split_sequences_with_asterisk, columns=["amino_acid_sequence"])
    
    # Adding the back translated dna sequences to the data frame.
    df_sequences['nt_sequence'] = df_sequences['amino_acid_sequence'].apply(lambda aa_seq: translate_to_dna_with_all_options(aa_seq)[0])
    
    df_sequences["overlap_seq"] = output_forward_reverse
    
    #here, we are attaching the overlap sequence to the known coding sequence, generated above. It is
    #added at the location based on the length of the overlap_seq
    def modify_sequence(df):
        # Check if 'overlap_seq' column exists in the DataFrame
        if 'overlap_seq' not in df.columns:
            raise ValueError("DataFrame must contain 'overlap_seq' column.")
    
        # Calculate the length of the overlap sequence
        df['overlap_length'] = df['overlap_seq'].apply(len)
    
        # Modify the sequences
        df['modified_sequence'] = df.apply(lambda row: row['nt_sequence'][:-row['overlap_length']] + row['overlap_seq'], axis=1)
    
        return df
    
    modify_sequence(df_sequences)
    
    # Custom function to translate using Seq
    def translate_with_seq(sequence):
        coding_dna = Seq(sequence)
        return str(coding_dna.translate())
    
    # Apply the custom function to the 'nt_sequence' column
    df_sequences['translated_sequence'] = df_sequences['modified_sequence'].apply(translate_with_seq)
    
    #print(compare_aa)
    return(df_sequences)


def align_sequences(seq1, seq2):
    # Perform the alignment using the Needleman-Wunsch algorithm
    alignments = pairwise2.align.globalxx(seq1, seq2)
    
    # Find the highest alignment score
    max_score = float('-inf')
    for a in alignments:
        score = a[2]
        if score > max_score:
            max_score = score
    
    return max_score

#defining this align_sequences_identity function to check pariwise identity matching, since the globalxx approach allows sequence shifting, 
#which is problematic in the partial-matching predictions.
def align_sequences_identity(seq1, seq2):
    # Ensure sequences are of the same length
    if len(seq1) != len(seq2):
        raise ValueError("Sequences must be of the same length for 1:1 alignment")

    # Calculate the alignment score by comparing each character
    score = sum(1 for a, b in zip(seq1, seq2) if a == b)
    
    return score

################################
# Function to generate partially matching convergent overlap sequences
################################

def find_partially_matching_sequence(model_to_predict, formatted_aa_seq, max_attempts, alignment_threshold_1, alignment_threshold_2, blosum_threshold_1, blosum_threshold_2, _counter=[0]):
    _counter[0] += 1

    print(f"Parsing row {_counter[0]}...")

    attempts = 0
    match_found = False
    matching_dataframe = None
    
    while attempts < max_attempts:
        attempts += 1

        try:
            result_df = predict_overlapping_sequence(model_to_predict, formatted_aa_seq, concat_tokenizer, overlap_tokenizer)
        except CodonTable.TranslationError as e:
            print(f"Error: {e}")
             # Print the "overlap_length" at the end of each iteration
            print(f"Attempt {attempts}... Overlap Length: {matching_dataframe['overlap_length'].iloc[0] if matching_dataframe is not None else 'N/A'}")
            continue
        
        # Calculate the max_diff variable
        #max_diff = round((result_df['overlap_length'].iloc[0]/3) * 0.3)
        
        # Perform alignment on full sequences
        #align_score_1 = align_sequences(result_df['amino_acid_sequence'].iloc[0], result_df['translated_sequence'].iloc[0])
        #align_score_2 = align_sequences(result_df['amino_acid_sequence'].iloc[1], result_df['translated_sequence'].iloc[1])
        
        # Calculate the new overlap length based on the provided conditions
        overlap_length = result_df['overlap_length'].iloc[0]

        if overlap_length % 3 == 0:
            overlap_length_3 = overlap_length / 3
        else:
            overlap_length_3 = (overlap_length + 1) / 3

        # Round the result to the nearest whole number
        overlap_length_3 = int(round(overlap_length_3, 0))
        
        # Truncate the sequences based on overlap_length_3
        truncated_aa_seq_1 = result_df['amino_acid_sequence'].iloc[0][-overlap_length_3:]
        truncated_trans_seq_1 = result_df['translated_sequence'].iloc[0][-overlap_length_3:]
        
        truncated_aa_seq_2 = result_df['amino_acid_sequence'].iloc[1][-overlap_length_3:]
        truncated_trans_seq_2 = result_df['translated_sequence'].iloc[1][-overlap_length_3:]

        blosum62_sim_score_1 = calculate_blosum62_similarity(truncated_aa_seq_1, truncated_trans_seq_1)
        blosum62_sim_score_2 = calculate_blosum62_similarity(truncated_aa_seq_2, truncated_trans_seq_2)

        blosum62_sim_score_1_ratio = (calculate_blosum62_similarity(truncated_aa_seq_1, truncated_trans_seq_1)/calculate_blosum62_similarity(truncated_aa_seq_1, truncated_aa_seq_1))
        blosum62_sim_score_2_ratio = (calculate_blosum62_similarity(truncated_aa_seq_2, truncated_trans_seq_2)/calculate_blosum62_similarity(truncated_aa_seq_2, truncated_aa_seq_2))

        protsub_sim_score_1 = calculate_protsub_similarity(truncated_aa_seq_1, truncated_trans_seq_1)
        protsub_sim_score_2 = calculate_protsub_similarity(truncated_aa_seq_2, truncated_trans_seq_2)
        
        # Perform new alignment on truncated sequences
        truncated_align_score_1 = align_sequences_identity(truncated_aa_seq_1, truncated_trans_seq_1) / overlap_length_3
        truncated_align_score_2 = align_sequences_identity(truncated_aa_seq_2, truncated_trans_seq_2) / overlap_length_3
        
        # Add new alignment scores to the DataFrame
        result_df['truncated_norm_align_score_1'] = round(truncated_align_score_1, 2)
        result_df['truncated_norm_align_score_2'] = round(truncated_align_score_2, 2)

        # Add blosum62 similarity scores to the DataFrame
        # Add new alignment scores to the DataFrame
        result_df['blosum62_sim_score_1'] = round(blosum62_sim_score_1, 2)
        result_df['blosum62_sim_score_1_len_norm'] = round(blosum62_sim_score_1 / overlap_length_3, 2)

        result_df['blosum62_sim_score_2'] = round(blosum62_sim_score_2, 2)
        result_df['blosum62_sim_score_2_len_norm'] = round(blosum62_sim_score_2 / overlap_length_3, 2)

        result_df['blosum62_sim_score_1_ratio'] = round(blosum62_sim_score_1_ratio, 2)
        result_df['blosum62_sim_score_2_ratio'] = round(blosum62_sim_score_2_ratio, 2)

        # Combine the two truncated alignment scores into one column
        result_df['blosum62_combined_scores'] = result_df.apply(
            lambda row: f"{row['blosum62_sim_score_1']} / {row['blosum62_sim_score_2']}",
            axis=1
        )

        # Combine the two truncated alignment scores into one column
        result_df['blosum62_combined_normalized_scores'] = result_df.apply(
            lambda row: f"{row['blosum62_sim_score_1_len_norm']} / {row['blosum62_sim_score_2_len_norm']}",
            axis=1
        )

        # Combine the two truncated alignment scores into one column
        result_df['blosum62_combined_ratio_scores'] = result_df.apply(
            lambda row: f"{row['blosum62_sim_score_1_ratio']} / {row['blosum62_sim_score_2_ratio']}",
            axis=1
        )

        # Combine the two truncated alignment scores into one column
        result_df['truncated_combined_scores'] = result_df.apply(
            lambda row: f"{row['truncated_norm_align_score_1']} / {row['truncated_norm_align_score_2']}",
            axis=1
        )

        # Add protsub similarity scores to the DataFrame
        # Add new alignment scores to the DataFrame
        result_df['protsub_sim_score_1'] = round(protsub_sim_score_1, 2)
        result_df['protsub_sim_score_1_len_norm'] = round(protsub_sim_score_1 / overlap_length_3, 2)

        result_df['protsub_sim_score_2'] = round(protsub_sim_score_2, 2)
        result_df['protsub_sim_score_2_len_norm'] = round(protsub_sim_score_2 / overlap_length_3, 2)

        # Combine the two truncated alignment scores into one column
        result_df['protsub_combined_scores'] = result_df.apply(
            lambda row: f"{row['protsub_sim_score_1']} / {row['protsub_sim_score_2']}",
            axis=1
        )

        # Combine the two truncated alignment scores into one column
        result_df['protsub_combined_normalized_scores'] = result_df.apply(
            lambda row: f"{row['protsub_sim_score_1_len_norm']} / {row['protsub_sim_score_2_len_norm']}",
            axis=1
        )

        #update me to set the alignment lower boundary desired
        align_limit_1 = alignment_threshold_1
        align_limit_2 = alignment_threshold_2
        blosum_limit_1 = blosum_threshold_1
        blosum_limit_2 = blosum_threshold_2

        if (truncated_align_score_1 >= align_limit_1 and
            truncated_align_score_2 >= align_limit_2 and
            result_df['blosum62_sim_score_1_len_norm'].iloc[0] >= blosum_limit_1 and
            result_df['blosum62_sim_score_2_len_norm'].iloc[0] >= blosum_limit_2 and
            result_df['translated_sequence'].iloc[0].count('*') == 1 and
            result_df['translated_sequence'].iloc[1].count('*') == 1 and
            result_df['translated_sequence'].iloc[0].endswith('*') and
            result_df['translated_sequence'].iloc[1].endswith('*')):
            match_found = True
            matching_dataframe = result_df
            break

    if match_found:
        # Print the combined truncated alignment scores
        combined_scores = matching_dataframe['truncated_combined_scores'].iloc[0]
        #combined_blosum_scores = matching_dataframe['blosum62_combined_scores'].iloc[0]
        combined_blosum_len_norm_scores = matching_dataframe['blosum62_combined_normalized_scores'].iloc[0]
        #combined_protsub_scores = matching_dataframe['protsub_combined_scores'].iloc[0]
        combined_protsub_len_norm_scores = matching_dataframe['protsub_combined_normalized_scores'].iloc[0]
        blosum62_combined_ratio_scores = matching_dataframe['blosum62_combined_ratio_scores'].iloc[0]
        print(f"Attempt {attempts}... Overlap Length: {matching_dataframe['overlap_length'].iloc[0] if matching_dataframe is not None else 'N/A'}")
        print(f"Truncated Combined Scores: {combined_scores}")
        #print(f"Blosum62 Similarity Scores: {combined_blosum_scores}")
        print(f"Blosum62 Length Normalized Similarity Scores: {combined_blosum_len_norm_scores}")
        print(f"Blosum62 Length Normalized Similarity Ratio Scores: {blosum62_combined_ratio_scores}")
        #print(f"ProtSub Similarity Scores: {combined_protsub_scores}")
        print(f"ProtSub Length Normalized Similarity Scores: {combined_protsub_len_norm_scores}")
        return matching_dataframe
    
    else:
        print("There is no predicted significant overlap")
        return None


def process_sequences(aa_seq_1, aa_seq_2):

    if len(aa_seq_1) < 103:
        return "Error: sequence_1 is not long enough to process. The minimum length is 103 amino acids."
    if len(aa_seq_2) < 103:
        return "Error: sequence_2 is not long enough to process. The minimum length is 103 amino acids."

    # Keep only the final 104 amino acids of each sequence
    aa_seq_1_trimmed = aa_seq_1[-104:]
    aa_seq_2_trimmed = aa_seq_2[-104:]

    # Replace the first amino acid with an 'M'
    aa_seq_1_processed = 'M' + aa_seq_1_trimmed[1:]
    aa_seq_2_processed = 'M' + aa_seq_2_trimmed[1:]

    # Add an asterisk after each sequence
    aa_seq_1_processed += '*'
    aa_seq_2_processed += '*'

    # Concatenate the two sequences
    concatenated_seq = aa_seq_1_processed + aa_seq_2_processed

    # Add a space between every character
    final_seq = ' '.join(concatenated_seq)

    # Return the processed sequence
    return final_seq

def translate_to_dna_with_all_options(aa_sequence: str) -> str:
    
    # Codons and their frequencies for each amino acid based on the E. coli table
    back_translation_code_with_all_options = {
        'A': [('GCG', 0.27), ('GCT', 0.26), ('GCC', 0.26), ('GCA', 0.21)],
        'C': [('TGC', 0.53), ('TGT', 0.47)],
        'D': [('GAT', 0.63), ('GAC', 0.37)],
        'E': [('GAA', 0.68), ('GAG', 0.32)],
        'F': [('TTT', 0.58), ('TTC', 0.42)],
        'G': [('GGC', 0.35), ('GGT', 0.32), ('GGG', 0.25), ('GGA', 0.08)],
        'H': [('CAT', 0.56), ('CAC', 0.44)],
        'I': [('ATT', 0.48), ('ATC', 0.39), ('ATA', 0.14)],
        'K': [('AAA', 0.74), ('AAG', 0.26)],
        'L': [('CTG', 0.43), ('CTT', 0.13), ('CTC', 0.13), ('TTA', 0.14), ('CTA', 0.07), ('TTG', 0.13)],
        'M': [('ATG', 1.00)],
        'N': [('AAC', 0.60), ('AAT', 0.40)],
        'P': [('CCG', 0.52), ('CCA', 0.19), ('CCT', 0.16), ('CCC', 0.13)],
        'Q': [('CAG', 0.66), ('CAA', 0.34)],
        'R': [('CGT', 0.36), ('CGC', 0.36), ('CGG', 0.11), ('AGA', 0.08), ('AGG', 0.05), ('CGA', 0.04)],
        'S': [('AGC', 0.24), ('TCC', 0.24), ('TCT', 0.17), ('TCG', 0.15), ('TCA', 0.14), ('AGT', 0.15)],
        'T': [('ACC', 0.36), ('ACA', 0.28), ('ACG', 0.25), ('ACT', 0.11)],
        'V': [('GTG', 0.46), ('GTT', 0.28), ('GTC', 0.15), ('GTA', 0.11)],
        'W': [('TGG', 1.00)],
        'Y': [('TAT', 0.59), ('TAC', 0.41)],
        '*': [('TAA', 0.61), ('TGA', 0.30), ('TAG', 0.09)]
    }

    def choose_codon_based_on_frequency(codons):
  
        # Extract codon names and their frequencies
        codon_names = [codon for codon, _ in codons]
        codon_freqs = [freq for _, freq in codons]

        # Normalize the frequencies to ensure they sum up to 1
        total_frequency = sum(codon_freqs)
        normalized_freqs = [freq / total_frequency for freq in codon_freqs]

        # Randomly select a codon based on the frequency distribution
        chosen_codon = np.random.choice(codon_names, p=normalized_freqs)

        return chosen_codon

    # For each amino acid in the sequence, choose the most probable codon
    chosen_codons = [choose_codon_based_on_frequency(back_translation_code_with_all_options[aa]) for aa in aa_sequence]

    # Combine the chosen codons to form the nucleotide sequence
    nucleotide_sequence = ''.join(chosen_codons)

    return nucleotide_sequence


# Function to integrate modified sequence into original sequence
def integrate_modified_sequence(original_dna, modified_dna):
    # Remove the terminal xx nucleotides from the original sequence
    trimmed_original_dna = original_dna[:-309] # this should be max nt length, minus 6, since the backtranslated sequence does not have stop codon DNA seqs
    # Remove the first three nucleotides from the modified sequence
    trimmed_modified_dna = modified_dna[3:]
    # Integrate the modified sequence
    integrated_sequence = trimmed_original_dna + trimmed_modified_dna
    return integrated_sequence

def is_dna_sequence(sequence: str) -> bool:
    """
    Determine if a sequence is a DNA sequence.
    A DNA sequence should only contain A, T, C, G (and sometimes N).
    """
    return all(char in 'ATCGNatcgn' for char in sequence)

def load_model(model_path, vocab_size, embedding_dim, num_blocks, num_heads, ffn_dim, max_length, dropout_rate):
    # Define and load model architecture
    model = TransformerModel(
        vocab_size=vocab_size, embedding_dim=embedding_dim, num_blocks=num_blocks,
        num_heads=num_heads, ffn_dim=ffn_dim, max_length=max_length, dropout_rate=dropout_rate
    )
    model.load_state_dict(torch.load(model_path))
    model.eval()
    return model.to(torch.device('cuda' if torch.cuda.is_available() else 'cpu'))

################################
# Inference function, incorporating the model inference, sequence processing, and dataset prepration processes
################################

def run_inference_for_models(model_numbers, seq_1, seq_2, max_attempts_first_pass, max_attempts_second_pass, first_pass_alignment_threshold_1, first_pass_alignment_threshold_2, second_pass_alignment_threshold_1, second_pass_alignment_threshold_2, first_pass_blosum_threshold_1, first_pass_blosum_threshold_2, second_pass_blosum_threshold_1, second_pass_blosum_threshold_2, first_pass_iterations, second_pass_iterations):
    
    if is_dna_sequence(seq_1) and is_dna_sequence(seq_2):
        # Input sequences are DNA, translate them to amino acids
        aa_seq_1 = str(Seq(seq_1).translate())[:-1]
        aa_seq_2 = str(Seq(seq_2).translate())[:-1]
      
    else:
        # Input sequences are amino acids, use them directly
        aa_seq_1 = seq_1
        aa_seq_2 = seq_2
        # Back-translate amino acid sequences to DNA sequences
        back_translated_aa_seq_1 = str(translate_to_dna_with_all_options(aa_seq_1))
        back_translated_aa_seq_2 = str(translate_to_dna_with_all_options(aa_seq_2))

    concatenated_sequence = process_sequences(aa_seq_1, aa_seq_2)

    # Check if the result is an error message or the processed sequence, and print accordingly
    if "Error" in concatenated_sequence:
        print(concatenated_sequence)
    else:
        print("The input amino acid sequences have been processed and concatenated: ")
        print(concatenated_sequence)

    formatted_aa_seq = concatenated_sequence

    # Swap the order of the sequence for the next iteration, preserving the asterisk
    parts = formatted_aa_seq.split('*')

    formatted_aa_seq_1 = f"{parts[0].strip()} * {parts[1].strip()} *"
    formatted_aa_seq_2 = f"{parts[1].strip()} * {parts[0].strip()} *"
    
    all_results = []
    
    # First pass: Run the initial inference
    for model_num in model_numbers:
        model_path = model_data.loc[model_data['overlap_length'] == model_num, 'model_pth_location'].values[0]
        model = torch.load(model_path, weights_only = False)
    
        print("################################################################################################")
        print('Attempting to predict overlaps of length: ' + str(model_num))
        print('This will be run ' + str(first_pass_iterations) + " time(s), and then repeated after switching primary and secondary sequence order.")
        print("################################################")

        # Run the function for the first formatted sequence
        for i in range(first_pass_iterations):
            result_df = find_partially_matching_sequence(model, formatted_aa_seq_1, max_attempts_first_pass, first_pass_alignment_threshold_1, first_pass_alignment_threshold_2, first_pass_blosum_threshold_1, first_pass_blosum_threshold_2)

            if result_df is not None:
                all_results.append({
                    'model_number': model_num,
                    'pass': 'first',
                    'iteration': i + 1,
                    'modified_sequence_1': result_df['modified_sequence'].iloc[0],
                    'modified_sequence_2': result_df['modified_sequence'].iloc[1],
                    'overlap_length': result_df['overlap_length'].iloc[0],
                    'translated_aa_seq_1': result_df['translated_sequence'].iloc[0],
                    'translated_aa_seq_2': result_df['translated_sequence'].iloc[1],
                    'truncated_norm_align_score_1': round(result_df['truncated_norm_align_score_1'].iloc[0], 2),
                    'truncated_norm_align_score_2': round(result_df['truncated_norm_align_score_2'].iloc[0], 2),
                    'truncated_norm_align_avg_score': round((result_df['truncated_norm_align_score_1'].iloc[0] + result_df['truncated_norm_align_score_2'].iloc[0])/2, 2),
                    'blosum62_sim_score_1': round(result_df['blosum62_sim_score_1'].iloc[0], 2),
                    'blosum62_sim_score_2': round(result_df['blosum62_sim_score_2'].iloc[0], 2),
                    'blosum_sim_score_len_norm_1': round(result_df['blosum62_sim_score_1_len_norm'].iloc[0], 2),
                    'blosum_sim_score_len_norm_2': round(result_df['blosum62_sim_score_2_len_norm'].iloc[0], 2),
                    'blosum62_sum_score': round(result_df['blosum62_sim_score_1'].iloc[0] + result_df['blosum62_sim_score_2'].iloc[0], 2),
                    'protsub_sim_score_1': round(result_df['protsub_sim_score_1'].iloc[0], 2),
                    'protsub_sim_score_2': round(result_df['protsub_sim_score_2'].iloc[0], 2),
                    'protsub_sim_score_len_norm_1': round(result_df['protsub_sim_score_1_len_norm'].iloc[0], 2),
                    'protsub_sim_score_len_norm_2': round(result_df['protsub_sim_score_2_len_norm'].iloc[0], 2),
                    'protsub_sum_score': round(result_df['protsub_sim_score_1'].iloc[0] + result_df['protsub_sim_score_2'].iloc[0], 2)
                })

        # Run the function for the second formatted sequence
        for i in range(first_pass_iterations):
            result_df = find_partially_matching_sequence(model, formatted_aa_seq_2, max_attempts_first_pass, first_pass_alignment_threshold_2, first_pass_alignment_threshold_1, first_pass_blosum_threshold_2, first_pass_blosum_threshold_1)

            if result_df is not None:
                all_results.append({
                    'model_number': model_num,
                    'pass': 'first',
                    'iteration': i + 1,
                    'modified_sequence_1': result_df['modified_sequence'].iloc[1],
                    'modified_sequence_2': result_df['modified_sequence'].iloc[0],
                    'overlap_length': result_df['overlap_length'].iloc[0],
                    'translated_aa_seq_1': result_df['translated_sequence'].iloc[1],
                    'translated_aa_seq_2': result_df['translated_sequence'].iloc[0],
                    'truncated_norm_align_score_1': round(result_df['truncated_norm_align_score_2'].iloc[0], 2),
                    'truncated_norm_align_score_2': round(result_df['truncated_norm_align_score_1'].iloc[0], 2),
                    'truncated_norm_align_avg_score': round((result_df['truncated_norm_align_score_1'].iloc[0] + result_df['truncated_norm_align_score_2'].iloc[0])/2, 2),
                    'blosum62_sim_score_1': round(result_df['blosum62_sim_score_2'].iloc[0], 2),
                    'blosum62_sim_score_2': round(result_df['blosum62_sim_score_1'].iloc[0], 2),
                    'blosum_sim_score_len_norm_1': round(result_df['blosum62_sim_score_2_len_norm'].iloc[0], 2),
                    'blosum_sim_score_len_norm_2': round(result_df['blosum62_sim_score_1_len_norm'].iloc[0], 2),
                    'blosum62_sum_score': round(result_df['blosum62_sim_score_1'].iloc[0] + result_df['blosum62_sim_score_2'].iloc[0], 2),
                    'protsub_sim_score_1': round(result_df['protsub_sim_score_2'].iloc[0], 2),
                    'protsub_sim_score_2': round(result_df['protsub_sim_score_1'].iloc[0], 2),
                    'protsub_sim_score_len_norm_1': round(result_df['protsub_sim_score_2_len_norm'].iloc[0], 2),
                    'protsub_sim_score_len_norm_2': round(result_df['protsub_sim_score_1_len_norm'].iloc[0], 2),
                    'protsub_sum_score': round(result_df['protsub_sim_score_1'].iloc[0] + result_df['protsub_sim_score_2'].iloc[0], 2)
                })

    # Create a DataFrame from the collected results
    results_df = pd.DataFrame(all_results)

    # Filter models with alignment scores above the threshold
    filtered_models = results_df[
        (results_df['truncated_norm_align_score_1'] >= first_pass_alignment_threshold_1) &
        (results_df['truncated_norm_align_score_2'] >= first_pass_alignment_threshold_2) &
        (results_df['overlap_length'] >= results_df['model_number'].apply(math.floor))
    ]

    # Ensure only one row per model number is included
    filtered_models = filtered_models.drop_duplicates(subset=['model_number'])

    if filtered_models.empty:
        print("No models meet the alignment threshold.")
        return None

    final_results = []

    # Second pass: Re-run inference for models that meet the threshold
    for _, row in filtered_models.iterrows():
        model_num = row['model_number']
        model = torch.load(model_data.loc[model_data['overlap_length'] == model_num, 'model_pth_location'].values[0], weights_only = False)

        print("################################################################################################")
        print('Attempting to predict overlaps of length: ' + str(model_num))
        print('This will be run ' + str(second_pass_iterations) + " time(s), and then repeated after switching primary and secondary sequence order.")
        print("################################################")

        for i in range(second_pass_iterations):
            result_df = find_partially_matching_sequence(model, formatted_aa_seq_1, max_attempts_second_pass, second_pass_alignment_threshold_1, second_pass_alignment_threshold_2, second_pass_blosum_threshold_1, second_pass_blosum_threshold_2)
            if result_df is not None:
                final_results.append({
                    'model_number': model_num,
                    'pass': 'second',
                    'iteration': i + 1,
                    'modified_sequence_1': result_df['modified_sequence'].iloc[0],
                    'modified_sequence_2': result_df['modified_sequence'].iloc[1],
                    'overlap_length': result_df['overlap_length'].iloc[0],
                    'translated_aa_seq_1': result_df['translated_sequence'].iloc[0],
                    'translated_aa_seq_2': result_df['translated_sequence'].iloc[1],
                    'truncated_norm_align_score_1': round(result_df['truncated_norm_align_score_1'].iloc[0], 2),
                    'truncated_norm_align_score_2': round(result_df['truncated_norm_align_score_2'].iloc[0], 2),
                    'truncated_norm_align_avg_score': round((result_df['truncated_norm_align_score_1'].iloc[0] + result_df['truncated_norm_align_score_2'].iloc[0])/2, 2),
                    'blosum62_sim_score_1': round(result_df['blosum62_sim_score_1'].iloc[0], 2),
                    'blosum62_sim_score_2': round(result_df['blosum62_sim_score_2'].iloc[0], 2),
                    'blosum_sim_score_len_norm_1': round(result_df['blosum62_sim_score_1_len_norm'].iloc[0], 2),
                    'blosum_sim_score_len_norm_2': round(result_df['blosum62_sim_score_2_len_norm'].iloc[0], 2),
                    'blosum62_sum_score': round(result_df['blosum62_sim_score_1'].iloc[0] + result_df['blosum62_sim_score_2'].iloc[0], 2),
                    'protsub_sim_score_1': round(result_df['protsub_sim_score_1'].iloc[0], 2),
                    'protsub_sim_score_2': round(result_df['protsub_sim_score_2'].iloc[0], 2),
                    'protsub_sim_score_len_norm_1': round(result_df['protsub_sim_score_1_len_norm'].iloc[0], 2),
                    'protsub_sim_score_len_norm_2': round(result_df['protsub_sim_score_2_len_norm'].iloc[0], 2),
                    'protsub_sum_score': round(result_df['protsub_sim_score_1'].iloc[0] + result_df['protsub_sim_score_2'].iloc[0], 2)
                })
        
            result_df = find_partially_matching_sequence(model, formatted_aa_seq_2, max_attempts_second_pass, second_pass_alignment_threshold_2, second_pass_alignment_threshold_1, second_pass_blosum_threshold_2, second_pass_blosum_threshold_1)
            if result_df is not None:
                final_results.append({
                    'model_number': model_num,
                    'pass': 'second',
                    'iteration': i + 1,
                    'modified_sequence_1': result_df['modified_sequence'].iloc[1],
                    'modified_sequence_2': result_df['modified_sequence'].iloc[0],
                    'overlap_length': result_df['overlap_length'].iloc[0],
                    'translated_aa_seq_1': result_df['translated_sequence'].iloc[1],
                    'translated_aa_seq_2': result_df['translated_sequence'].iloc[0],
                    'truncated_norm_align_score_1': round(result_df['truncated_norm_align_score_2'].iloc[0], 2),
                    'truncated_norm_align_score_2': round(result_df['truncated_norm_align_score_1'].iloc[0], 2),
                    'truncated_norm_align_avg_score': round((result_df['truncated_norm_align_score_1'].iloc[0] + result_df['truncated_norm_align_score_2'].iloc[0])/2, 2),
                    'blosum62_sim_score_1': round(result_df['blosum62_sim_score_2'].iloc[0], 2),
                    'blosum62_sim_score_2': round(result_df['blosum62_sim_score_1'].iloc[0], 2),
                    'blosum_sim_score_len_norm_1': round(result_df['blosum62_sim_score_2_len_norm'].iloc[0], 2),
                    'blosum_sim_score_len_norm_2': round(result_df['blosum62_sim_score_1_len_norm'].iloc[0], 2),
                    'blosum62_sum_score': round(result_df['blosum62_sim_score_1'].iloc[0] + result_df['blosum62_sim_score_2'].iloc[0], 2),
                    'protsub_sim_score_1': round(result_df['protsub_sim_score_2'].iloc[0], 2),
                    'protsub_sim_score_2': round(result_df['protsub_sim_score_1'].iloc[0], 2),
                    'protsub_sim_score_len_norm_1': round(result_df['protsub_sim_score_2_len_norm'].iloc[0], 2),
                    'protsub_sim_score_len_norm_2': round(result_df['protsub_sim_score_1_len_norm'].iloc[0], 2),
                    'protsub_sum_score': round(result_df['protsub_sim_score_1'].iloc[0] + result_df['protsub_sim_score_2'].iloc[0], 2)
                })

    # Combine all final results into a single DataFrame
    final_results_df = pd.DataFrame(final_results + all_results)

    
    if is_dna_sequence(seq_1) and is_dna_sequence(seq_2):

            # Add columns for integrated sequences
        final_results_df['integrated_seq_1'] = final_results_df.apply(
            lambda row: integrate_modified_sequence(seq_1, row['modified_sequence_1']), axis=1)

        final_results_df['integrated_seq_2'] = final_results_df.apply(
            lambda row: integrate_modified_sequence(seq_2, row['modified_sequence_2']), axis=1)

        final_results_df['translated_integrated_seq_1'] = final_results_df['integrated_seq_1'].apply(
            lambda seq: str(Seq(seq).translate()))
        
        final_results_df['translated_integrated_seq_2'] = final_results_df['integrated_seq_2'].apply(
            lambda seq: str(Seq(seq).translate()))
    
    else:

        # Add columns for integrated sequences
        final_results_df['integrated_seq_1'] = final_results_df.apply(
            lambda row: integrate_modified_sequence(back_translated_aa_seq_1, row['modified_sequence_1']), axis=1)

        final_results_df['integrated_seq_2'] = final_results_df.apply(
            lambda row: integrate_modified_sequence(back_translated_aa_seq_2, row['modified_sequence_2']), axis=1)

        final_results_df['translated_integrated_seq_1'] = final_results_df['integrated_seq_1'].apply(
            lambda seq: str(Seq(seq).translate()))
        
        final_results_df['translated_integrated_seq_2'] = final_results_df['integrated_seq_2'].apply(
            lambda seq: str(Seq(seq).translate()))
    
    # Print the final results
    print(final_results_df)
    return final_results_df

################################
#### Functions to match specific AAs between 2 sequences, for each specific AA set within brackets
################################

def get_bracket_positions(sequence_with_brackets):
    """
    Extract positions of amino acids within brackets in sequence1 and map them to sequence2.

    Args:
        sequence_with_brackets (str): The amino acid sequence with bracketed sections.

    Returns:
        list of tuples: Each tuple contains the start position in sequence2 and the bracketed amino acids.
    """
    bracketed_amino_acids = []
    pos_with_brackets = 0  # Position in sequence1 (with brackets)
    pos_without_brackets = 0  # Position in sequence2 (without brackets)

    while pos_with_brackets < len(sequence_with_brackets):
        if sequence_with_brackets[pos_with_brackets] == '[':
            pos_with_brackets += 1  # Skip '['
            bracket_start_seq2 = pos_without_brackets  # Record start position in sequence2
            bracket_content = []

            # Collect all amino acids within the brackets
            while pos_with_brackets < len(sequence_with_brackets) and sequence_with_brackets[pos_with_brackets] != ']':
                bracket_content.append(sequence_with_brackets[pos_with_brackets])
                pos_with_brackets += 1
                pos_without_brackets += 1  # Advance sequence2 position as brackets are not present

            if pos_with_brackets >= len(sequence_with_brackets):
                raise ValueError("Unmatched '[' in sequence.")

            pos_with_brackets += 1  # Skip ']'
            bracketed_amino_acids.append((bracket_start_seq2, ''.join(bracket_content)))
        else:
            # Regular amino acid, advance both positions
            pos_with_brackets += 1
            pos_without_brackets += 1

    return bracketed_amino_acids

def compare_sequences_aa_selected(sequence1_with_brackets, sequence2):
    """
    Compare specific bracketed sections in sequence1 with sequence2.

    Args:
        sequence1_with_brackets (str): First amino acid sequence with brackets.
        sequence2 (str): Second amino acid sequence without brackets.

    Returns:
        dict: A dictionary with start positions in sequence2 as keys and True/False for match/mismatch.
    """
    bracketed_amino_acids = get_bracket_positions(sequence1_with_brackets)
    results = {}

    for start, amino_acids in bracketed_amino_acids:
        # Extract amino acids from sequence2 for the corresponding positions
        seq2_amino_acids = sequence2[start:start + len(amino_acids)]
        results[start] = (seq2_amino_acids == amino_acids)  # True if match, False if not

    return results

def get_comparison_results(sequence1, sequence2):
    # Function to get comparison results between sequence1 and sequence2
    # This uses your existing compare_sequences function or a similar logic
    # For demonstration, replace this with the actual comparison logic
    comparison_results = compare_sequences_aa_selected(sequence1, sequence2)
    return comparison_results

def match_status(comparison_results):
    """
    Determines the match status based on the values in comparison_results.

    Args:
        comparison_results (dict): A dictionary with match results (True/False values).

    Returns:
        str: A message indicating whether all, some, or none of the comparisons matched.
    """
    if all(comparison_results.values()):
        return "Match"
    elif any(comparison_results.values()):
        return "Partial match"
    else:
        return "No match"

################################
# This function compares the secondary structures of the primary and secondary sequences, for the original and predictions
################################

def compare_sequences(seq1, seq2, pred1, pred2):
    """
    Compare two sequences with their respective predictions and score the matches.
    
    Args:
    - seq1: Original sequence 1 (string)
    - seq2: Original sequence 2 (string)
    - pred1: Predicted sequence 1 (string)
    - pred2: Predicted sequence 2 (string)
    
    Returns:
    - match_count: Number of positions where both sequences match their predictions
    - combined_score: Percentage score of combined matches relative to total positions
    - match_count_seq1: Number of positions where seq1 matches pred1
    - score_seq1: Percentage score of matches for seq1 relative to total positions
    - match_count_seq2: Number of positions where seq2 matches pred2
    - score_seq2: Percentage score of matches for seq2 relative to total positions
    - average_score_seq1_seq2: Mean average score of seq1 and seq2
    - abs_value: Absolute difference between score_seq1 and score_seq2
    """
    # Initialize counters
    total_positions = len(seq1)  # Assuming both sequences are the same length
    match_count = 0
    match_count_seq1 = 0
    match_count_seq2 = 0

    # Comparison loop
    for i in range(total_positions):
        if seq1[i] == pred1[i]:
            match_count_seq1 += 1
        if seq2[i] == pred2[i]:
            match_count_seq2 += 1
        if seq1[i] == pred1[i] and seq2[i] == pred2[i]:
            match_count += 1

    # Calculate scores
    combined_score = round(match_count / total_positions * 100, 2)
    score_seq1 = round(match_count_seq1 / total_positions * 100, 2)
    score_seq2 = round(match_count_seq2 / total_positions * 100, 2)
    average_score_seq1_seq2 = round((score_seq1 + score_seq2) / 2, 2)
    abs_value = round(abs(score_seq1 - score_seq2), 2)

    return (match_count, combined_score, match_count_seq1, score_seq1, 
            match_count_seq2, score_seq2, average_score_seq1_seq2, abs_value)

###########################
# Functions to predict secondary structures using S4Pred, batched
###########################

output_dir = "/content/convergent_overlaps_aa_change/s4pred/outputs"

# This is used for single AA sequence secondary structure predictions
def structure_prediction_wrapper(sequence):
    output_dir = "/content/convergent_overlaps_aa_change/s4pred/outputs"
    
    # Call the prediction function
    prediction = predict_secondary_structure(sequence, output_dir)
    
    # Print the sequence and its prediction
    print(f"Sequence: {sequence}")
    print(f"Predicted Structure: {prediction}")
    print("-" * 50)  # Separator for readability
    
    return prediction

# This is used for batched AA sequence secondary structure predictions
def batch_structure_prediction_wrapper(sequences, output_dir):
    # Call the batch prediction function
    predictions = predict_secondary_structure_batch(sequences, output_dir)
    
    # Return the list of predictions
    return predictions

#####################
# Function to re-calculate an alignment score based on the original sequence, using the predicted sequence from a re-run of inference (used to fine-tune the sequence)
#####################

# Function to define the alignment score if rerun is True.
def rerun_alignment_score(original_sequence, predicted_sequence, overlap_length):
    # Remove terminal asterisk if it exists
    if original_sequence.endswith('*'):
        original_sequence = original_sequence[:-1]
    if predicted_sequence.endswith('*'):
        predicted_sequence = predicted_sequence[:-1]

    # Calculate overlap length in amino acids
    if overlap_length % 3 == 0:
        overlap_length_3 = overlap_length / 3
    else:
        overlap_length_3 = (overlap_length + 1) / 3

    # Round the result to the nearest whole number
    overlap_length_3 = int(round(overlap_length_3, 0))
    
    # Truncate the sequences based on overlap_length_3
    truncated_aa_seq = original_sequence[-overlap_length_3:]
    truncated_trans_seq = predicted_sequence[-overlap_length_3:]

    # Perform new alignment on truncated sequences
    truncated_align_score = align_sequences_identity(truncated_aa_seq, truncated_trans_seq) / overlap_length_3

    return truncated_align_score



In [5]:
###########
## Model Data File
###########

model_data = pd.read_csv('/content/convergent_overlaps_aa_change/aa_change_model_set/overlap_length_models_aa_change_random_pairs_315nt_length_199_312_20241024_colab.csv')


In [6]:

###################
### EGFP - AmpR Sequences

# Starting sequences, but with specified AAs in brackets to assess for matches
sequence_1_aa_brackets = "MVSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFICTTGKLPVPWPTLVTTL[TYG]VQCFSRYPDHMKQHDFFKSAMPEGYVQERTIFFKDDGNYKTRAEVKFEGDTLVNRIELKGIDFKEDGNILGHKLEYNYNS[H]NVYIMADKQKNGIKVNFKIRHNIEQGTLQFAEHKQQPCPMVDRPVLVPESHHLS[T]QSALPYDPPEKTQQIVLF[E]LVAAAGFPLANNQLFD"
sequence_2_aa_brackets = "MSIQHFRVALIPFFAAFCLPVFAHPETLVKVKDAEDQLGARVGYIELDLNSGKILESFRPEERFPMM[S]TF[K]VLLCGAVLSRIDAGQEQLGRRIHYSQNDLVEYSPVTEKHLTDGMTVRELCSAAITMSDNTAANLLLTTIGGPKELTAFLHNMGDHVTRLDRW[E]PEL[N]EAIPNDERDTTMPVAMATTLRKLLTGELLTLASFNQIIDCLPAENVAGPLLRSALPAGWFIADKSG[A]GERGTSGVIPALGPDGQPSGMVVVYVPRTEACLDESNGEIAKLVACLVGHY"

#Start with 0.36 as threshold for all runs
#EGFP - AmpR proteins

sequence_1_original = "MVSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFICTTGKLPVPWPTLVTTLTYGVQCFSRYPDHMKQHDFFKSAMPEGYVQERTIFFKDDGNYKTRAEVKFEGDTLVNRIELKGIDFKEDGNILGHKLEYNYNSHNVYIMADKQKNGIKVNFKIRHNIEDGSVQLADHYQQNTPIGDGPVLLPDNHYLSTQSALSKDPNEKRDHMVLLEFVTAAGITLGMDELYK"
sequence_2_original = "MSIQHFRVALIPFFAAFCLPVFAHPETLVKVKDAEDQLGARVGYIELDLNSGKILESFRPEERFPMMSTFKVLLCGAVLSRIDAGQEQLGRRIHYSQNDLVEYSPVTEKHLTDGMTVRELCSAAITMSDNTAANLLLTTIGGPKELTAFLHNMGDHVTRLDRWEPELNEAIPNDERDTTMPVAMATTLRKLLTGELLTLASRQQLIDWMEADKVAGPLLRSALPAGWFIADKSGAGERGSRGIIAALGPDGKPSRIVVIYTTGSQATMDERNRQIAEIGASLIKHW"

concatenated_sequence_original = process_sequences(sequence_1_original, sequence_2_original)
concatenated_sequence_original

def split_sequence(sequence):
    # Remove spaces from the sequence
    cleaned_sequence = sequence.replace(" ", "")
    
    # Split the sequence at asterisks and remove any empty strings in case of multiple asterisks
    split_seqs = [seq for seq in cleaned_sequence.split('*') if seq]
    
    # Ensure only two sequences are returned
    if len(split_seqs) == 2:
        return split_seqs[0], split_seqs[1]
    else:
        raise ValueError("The sequence does not split cleanly into two parts with a single asterisk.")
        
ori_seq_1, ori_seq_2 = split_sequence(concatenated_sequence_original)

In [ ]:

ori_1_predicted_structure = structure_prediction_wrapper(ori_seq_1)
ori_2_predicted_structure = structure_prediction_wrapper(ori_seq_2)

In [8]:
model_numbers = [257]

In [10]:

first_pass_iterations = 1  # User-defined number of iterations for the first pass; this will filter out lengths where predictions do not meet the threshold, or that result in lengths shorter than the model target
second_pass_iterations = 5 # User-defined number of iterations for the second pass
first_pass_alignment_threshold_1 = 0.36  # User-defined alignment identity threshold first pass; applied to both sequences
first_pass_alignment_threshold_2 = 0.36 # User-defined alignment identity threshold first pass; applied to both sequences
second_pass_alignment_threshold_1 = 0.36 # User-defined alignment identity threshold second pass; applied to both sequences
second_pass_alignment_threshold_2 = 0.36  # User-defined alignment identity threshold second pass; applied to both sequences
first_pass_blosum_threshold_1 = 0  # User-defined alignment identity threshold first pass; applied to both sequences
first_pass_blosum_threshold_2 = 0  # User-defined alignment identity threshold first pass; applied to both sequences
second_pass_blosum_threshold_1 = 0 # User-defined alignment identity threshold second pass; applied to both sequences
second_pass_blosum_threshold_2 = 0 # User-defined alignment identity threshold second pass; applied to both sequences
max_attempts_first_pass = 325  # Maximum attempts for each sequence first pass
max_attempts_second_pass = 200 # Maximum attempts for each sequence second pass


In [ ]:

# Call the function
final_results_df = run_inference_for_models(model_numbers, sequence_1_original, sequence_2_original, max_attempts_first_pass, max_attempts_second_pass, first_pass_alignment_threshold_1, first_pass_alignment_threshold_2, second_pass_alignment_threshold_1, second_pass_alignment_threshold_2, first_pass_blosum_threshold_1, first_pass_blosum_threshold_2, second_pass_blosum_threshold_1, second_pass_blosum_threshold_2, first_pass_iterations, second_pass_iterations)


In [12]:
# Apply the batch function to the entire column of sequences
aa_sequences_1 = final_results_df["translated_aa_seq_1"].tolist()
predicted_structures_1 = batch_structure_prediction_wrapper(aa_sequences_1, output_dir)
# Assign the predicted structures back to the DataFrame
final_results_df["structure_prediction_1"] = predicted_structures_1

# Apply the batch function to the entire column of sequences
aa_sequences_2 = final_results_df["translated_aa_seq_2"].tolist()
predicted_structures_2 = batch_structure_prediction_wrapper(aa_sequences_2, output_dir)
# Assign the predicted structures back to the DataFrame
final_results_df["structure_prediction_2"] = predicted_structures_2
# Display the DataFrame with predictions


In [13]:

# Initialize lists to hold match counts and scores for each row
match_counts = []
combined_scores = []
match_counts_seq1 = []
scores_seq1 = []
match_counts_seq2 = []
scores_seq2 = []
average_scores_seq1_seq2 = []
absolute_values = []

# Iterate through each row in the DataFrame
for index, row in final_results_df.iterrows():
    pred1 = row["structure_prediction_1"]
    pred2 = row["structure_prediction_2"]
    
    # Call the updated compare_sequences function
    results = compare_sequences(
        ori_1_predicted_structure, ori_2_predicted_structure, pred1, pred2
    )
    
    # Unpack the results
    match_count, combined_score, match_count_seq1, score_seq1, match_count_seq2, score_seq2, average_score_seq1_seq2, abs_value = results
    
    # Append the results to the respective lists
    match_counts.append(match_count)
    combined_scores.append(combined_score)
    match_counts_seq1.append(match_count_seq1)
    scores_seq1.append(score_seq1)
    match_counts_seq2.append(match_count_seq2)
    scores_seq2.append(score_seq2)
    average_scores_seq1_seq2.append(average_score_seq1_seq2)
    absolute_values.append(abs_value)

# Add new columns to the DataFrame
final_results_df['match_count'] = match_counts
final_results_df['combined_score'] = combined_scores
final_results_df['match_count_seq1'] = match_counts_seq1
final_results_df['score_seq1'] = scores_seq1
final_results_df['match_count_seq2'] = match_counts_seq2
final_results_df['score_seq2'] = scores_seq2
final_results_df['average_score_seq1_seq2'] = average_scores_seq1_seq2
final_results_df['abs_value'] = absolute_values

# Calculate the mean average score by overlap_length (assuming overlap_length is a column in final_results_df)
mean_average_scores = final_results_df.groupby('overlap_length')['average_score_seq1_seq2'].mean().reset_index().round(2)
mean_average_scores.rename(columns={'average_score_seq1_seq2': 'mean_average_score_seq1_seq2'}, inplace=True)

# Merge the mean scores back to the original DataFrame
final_results_df = final_results_df.merge(mean_average_scores, on='overlap_length', how='left')

# Add the match-status for specified bracketed amino acids (defined above); if this is not needed (eg, if specified amino acids are not defined), do not run these lines

# Apply the comparison and match status functions to each row
final_results_df['comparison_results_1'] = final_results_df['translated_integrated_seq_1'].apply(
    lambda seq2: get_comparison_results(sequence_1_aa_brackets, seq2)
)

# Apply match status to each comparison result
final_results_df['match_status_1'] = final_results_df['comparison_results_1'].apply(match_status)

# Apply the comparison and match status functions to each row
final_results_df['comparison_results_2'] = final_results_df['translated_integrated_seq_2'].apply(
    lambda seq2: get_comparison_results(sequence_2_aa_brackets, seq2)
)

# Apply match status to each comparison result
final_results_df['match_status_2'] = final_results_df['comparison_results_2'].apply(match_status)


In [ ]:

final_results_df.to_csv('/mnt/d/RStuff/codon_overlap/aa_change_predictions/aa_pair_1_aa_change_315_range_199_312_egfp_ampr_20241107_test_notebook.csv', index=False)
final_results_df.to_excel('/mnt/d/RStuff/codon_overlap/aa_change_predictions/aa_pair_1_aa_change_315_range_199_312_egfp_ampr_20241107_test_notebook.xlsx', index=False)

In [14]:

########################################################################################################################
# Re-run/fine-tune the sequence(s)
########################################################################################################################

#Select high structural similarity sequence, then revert/change any desired amino acid sequences in one or both aa chains.
#Run sequence 1 and 2 as these modified version of the entire sequence with high structural similarity.
#Use a very high threshold, such that approximately only the number of changed sequences account for variation (eg, 0.95 for all runs)
#EGFP - AmpR -- Structurally similar predictions

sequence_1_modified = "MVSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFICTTGKLPVPWPTLVTTLTYGVQCFSRYPDHMKQHDFFKSAMPEGYVQERTIFFKDDGNYKTRAEVKFEGDTLVNRIELKGIDFKEDGNILGHKLEYNYNSHNVYIMVNKEKKGMRVNLKIRHNVDQGSLELAEYKQQKTPMGDGPALLPQKYHLSTQCALPNDPKENRQQVMLLEVVAAAGIPLGMNQLDD"
sequence_2_modified = "MSIQHFRVALIPFFAAFCLPVFAHPETLVKVKDAEDQLGARVGYIELDLNSGKILESFRPEERFPMMSTFKVLLCGAVLSRIDAGQEQLGRRIHYSQNDLVEYSPVTEKHLTDGMTVRELCSAAITMSDNTAANLLLTTIGGPKELTAFLHNMGDHVTRLDRWEPELNEAIPNDERDTTMPVAMATTLRKLLTGELLTLASLNHLIDSFQAEYLLRPLLRAALPAGWFIADKSGAGERGSSGIFAAAGPDRRPSGFFVVYTRRAQATLDQRYGEFSNLRACLFSLY"

concatenated_sequence_modified = process_sequences(sequence_1_modified, sequence_2_modified)
concatenated_sequence_modified

mod_seq_1, mod_seq_2 = split_sequence(concatenated_sequence_modified)

model_numbers = [257]


In [17]:

first_pass_iterations = 1  # User-defined number of iterations for the first pass; this will filter out lengths where predictions do not meet the threshold, or that result in lengths shorter than the model target
second_pass_iterations = 200 # User-defined number of iterations for the second pass
first_pass_alignment_threshold_1 = 0.50  # User-defined alignment identity threshold first pass; applied to both sequences
first_pass_alignment_threshold_2 = 0.50 # User-defined alignment identity threshold first pass; applied to both sequences
second_pass_alignment_threshold_1 = 0.50 # User-defined alignment identity threshold second pass; applied to both sequences
second_pass_alignment_threshold_2 = 0.50  # User-defined alignment identity threshold second pass; applied to both sequences
first_pass_blosum_threshold_1 = 0  # User-defined alignment identity threshold first pass; applied to both sequences
first_pass_blosum_threshold_2 = 0  # User-defined alignment identity threshold first pass; applied to both sequences
second_pass_blosum_threshold_1 = 0 # User-defined alignment identity threshold second pass; applied to both sequences
second_pass_blosum_threshold_2 = 0 # User-defined alignment identity threshold second pass; applied to both sequences
max_attempts_first_pass = 325  # Maximum attempts for each sequence first pass
max_attempts_second_pass = 200 # Maximum attempts for each sequence second pass

In [ ]:

# Call the function
final_results_df = run_inference_for_models(model_numbers, sequence_1_modified, sequence_2_modified, max_attempts_first_pass, max_attempts_second_pass, first_pass_alignment_threshold_1, first_pass_alignment_threshold_2, second_pass_alignment_threshold_1, second_pass_alignment_threshold_2, first_pass_blosum_threshold_1, first_pass_blosum_threshold_2, second_pass_blosum_threshold_1, second_pass_blosum_threshold_2, first_pass_iterations, second_pass_iterations)


In [19]:

######################################################################
##### If a rerun is performed, use this to add the alignment scores to the dataframe before saving
######################################################################

# Calculate the alignment scores for each original sequence and round to 2 decimal places
final_results_df['truncated_norm_align_score_rerun_to_original_1'] = final_results_df.apply(
    lambda row: round(rerun_alignment_score(ori_seq_1, row['translated_aa_seq_1'], row['overlap_length']), 2),
    axis=1
)

final_results_df['truncated_norm_align_score_rerun_to_original_2'] = final_results_df.apply(
    lambda row: round(rerun_alignment_score(ori_seq_2, row['translated_aa_seq_2'], row['overlap_length']), 2),
    axis=1
)

# Calculate the mean of the two scores, rounded to 2 decimal places
final_results_df['truncated_norm_align_avg_score_rerun_to_original'] = final_results_df.apply(
    lambda row: round((row['truncated_norm_align_score_rerun_to_original_1'] + row['truncated_norm_align_score_rerun_to_original_2']) / 2, 2),
    axis=1
)

#####################

In [21]:

# Apply the batch function to the entire column of sequences
aa_sequences_1 = final_results_df["translated_aa_seq_1"].tolist()
predicted_structures_1 = batch_structure_prediction_wrapper(aa_sequences_1, output_dir)
# Assign the predicted structures back to the DataFrame
final_results_df["structure_prediction_1"] = predicted_structures_1

# Apply the batch function to the entire column of sequences
aa_sequences_2 = final_results_df["translated_aa_seq_2"].tolist()
predicted_structures_2 = batch_structure_prediction_wrapper(aa_sequences_2, output_dir)
# Assign the predicted structures back to the DataFrame
final_results_df["structure_prediction_2"] = predicted_structures_2
# Display the DataFrame with predictions

In [ ]:
# Initialize lists to hold match counts and scores for each row
match_counts = []
combined_scores = []
match_counts_seq1 = []
scores_seq1 = []
match_counts_seq2 = []
scores_seq2 = []
average_scores_seq1_seq2 = []
absolute_values = []

# Iterate through each row in the DataFrame
for index, row in final_results_df.iterrows():
    pred1 = row["structure_prediction_1"]
    pred2 = row["structure_prediction_2"]
    
    # Call the updated compare_sequences function
    results = compare_sequences(
        ori_1_predicted_structure, ori_2_predicted_structure, pred1, pred2
    )
    
    # Unpack the results
    match_count, combined_score, match_count_seq1, score_seq1, match_count_seq2, score_seq2, average_score_seq1_seq2, abs_value = results
    
    # Append the results to the respective lists
    match_counts.append(match_count)
    combined_scores.append(combined_score)
    match_counts_seq1.append(match_count_seq1)
    scores_seq1.append(score_seq1)
    match_counts_seq2.append(match_count_seq2)
    scores_seq2.append(score_seq2)
    average_scores_seq1_seq2.append(average_score_seq1_seq2)
    absolute_values.append(abs_value)

# Add new columns to the DataFrame
final_results_df['match_count'] = match_counts
final_results_df['combined_score'] = combined_scores
final_results_df['match_count_seq1'] = match_counts_seq1
final_results_df['score_seq1'] = scores_seq1
final_results_df['match_count_seq2'] = match_counts_seq2
final_results_df['score_seq2'] = scores_seq2
final_results_df['average_score_seq1_seq2'] = average_scores_seq1_seq2
final_results_df['abs_value'] = absolute_values

# Calculate the mean average score by overlap_length (assuming overlap_length is a column in final_results_df)
mean_average_scores = final_results_df.groupby('overlap_length')['average_score_seq1_seq2'].mean().reset_index().round(2)
mean_average_scores.rename(columns={'average_score_seq1_seq2': 'mean_average_score_seq1_seq2'}, inplace=True)

# Merge the mean scores back to the original DataFrame
final_results_df = final_results_df.merge(mean_average_scores, on='overlap_length', how='left')

######################################################################
######################################################################
######################################################################

# Apply the comparison and match status functions to each row
final_results_df['comparison_results_1'] = final_results_df['translated_integrated_seq_1'].apply(
    lambda seq2: get_comparison_results(sequence_1_aa_brackets, seq2)
)

# Apply match status to each comparison result
final_results_df['match_status_1'] = final_results_df['comparison_results_1'].apply(match_status)

# Apply the comparison and match status functions to each row
final_results_df['comparison_results_2'] = final_results_df['translated_integrated_seq_2'].apply(
    lambda seq2: get_comparison_results(sequence_2_aa_brackets, seq2)
)

# Apply match status to each comparison result
final_results_df['match_status_2'] = final_results_df['comparison_results_2'].apply(match_status)

# Display the updated DataFrame
final_results_df.head()


In [23]:

### Saving data

final_results_df.to_csv('/mnt/d/RStuff/codon_overlap/aa_change_predictions/aa_pair_1_aa_change_315_range_199_312_egfp_ampr_20241108_257__test_notebook.csv', index=False)
final_results_df.to_excel('/mnt/d/RStuff/codon_overlap/aa_change_predictions/aa_pair_1_aa_change_315_range_199_312_egfp_ampr_20241108_257_test_notebook.xlsx', index=False)